# Charades · NB04 — Random-Walk Graph Visualization (Plotly)

Visualizes label propagation for **one window of one video**, in three panels that mirror the lecture figures: **(1) Input graph** (CDF-sparsified), **(2) Seeds on graph** (colored by class, labels in red), **(3) Output after RW** (every node colored by its single predicted class = argmax). Single-action illustration of the propagation mechanism. Uses the same stage functions as NB03 so the plotted graph is identical to the pipeline's. Requires `plotly` and `networkx`; reads `artifacts_charades/`.


In [1]:
import numpy as np, pickle
from pathlib import Path
import networkx as nx
import plotly.graph_objects as go

OUT_DIR=Path('./artifacts_charades')
FEAT_NORM='center'; SIGMA_MODE='median'; GAMMA=0.90; RW_STEPS=10   # match NB03 defaults

# choose ONE video and window to visualize
VIDEO='00SL4'      # clean video; try '00ZCA' for the densest
N=100              # frames in the window
WIN_START=0        # first frame of the window (0-based internal index)
LAYOUT_SEED=10      # fixes node positions so all 3 panels align
print('viz config:',dict(VIDEO=VIDEO,N=N,WIN_START=WIN_START,GAMMA=GAMMA,RW_STEPS=RW_STEPS))


viz config: {'VIDEO': '00SL4', 'N': 100, 'WIN_START': 0, 'GAMMA': 0.9, 'RW_STEPS': 10}


In [2]:
# ----- stage functions (identical to NB03) -----
def cosine_similarity(F):
    Fn=F/(np.linalg.norm(F,axis=1,keepdims=True)+1e-12); return Fn@Fn.T
def row_normalize(W): return W/(W.sum(1,keepdims=True)+1e-12)
def normalize_features(F):
    if FEAT_NORM=='none': return F
    F=F-F.mean(0,keepdims=True)
    if FEAT_NORM=='standardize': F=F/(F.std(0,keepdims=True)+1e-8)
    return F
def estimate_sigma(D):
    iu=np.triu_indices_from(D,k=1); d=D[iu]; s=np.median(d) if d.size else 1.0
    return float(s) if s>1e-6 else 1.0
def pdf_weights(F,sigma=None):
    S=cosine_similarity(F); D=1.0-S
    if sigma is None: sigma=estimate_sigma(D) if SIGMA_MODE=='median' else float(SIGMA_MODE)
    W=(1.0/(np.sqrt(2*np.pi)*sigma))*np.exp(-(D**2)/(2*sigma**2))
    return S,D,W,sigma
def cdf_sparsify(W,gamma=GAMMA):
    P=row_normalize(W); keep=np.zeros_like(W,bool)
    for i in range(W.shape[0]):
        o=np.argsort(-P[i]); cs=np.cumsum(P[i][o]); k=np.searchsorted(cs,gamma)+1; keep[i,o[:k]]=True
    return W*keep
def add_temporal_edges(A,wt=1.0):
    A=A.copy()
    for i in range(A.shape[0]-1): A[i,i+1]=max(A[i,i+1],wt); A[i+1,i]=max(A[i+1,i],wt)
    return A
def transition_matrix(A): return A/(A.sum(0,keepdims=True)+1e-12)
def random_walk(P,p0,steps):
    p=p0.copy()
    for _ in range(steps): p=P@p
    return p
def propagate(P,Y,seed_mask,steps=RW_STEPS):
    n,C=Y.shape; sc=np.zeros((n,C)); si=np.where(seed_mask)[0]
    for cc in range(C):
        cls=[i for i in si if Y[i,cc]>0]
        if not cls: continue
        p0=np.zeros(n); p0[cls]=1.0/len(cls); sc[:,cc]=random_walk(P,p0,steps)
    return sc
print('stage functions ready')


stage functions ready


In [3]:
# ----- load the video window, build the (sparsified) graph -----
names=pickle.load(open(OUT_DIR/'class_map.pkl','rb'))['names']
d=np.load(OUT_DIR/f'{VIDEO}_data.npz',allow_pickle=True)
feats=np.load(OUT_DIR/f'{VIDEO}_feats.npz')['feats']
Yall,seedsall=d['targets'],d['seeds']
e=min(WIN_START+N, feats.shape[0]); idx=np.arange(WIN_START,e)
F=normalize_features(feats[idx]); Yw=Yall[idx]; seedw=seedsall[idx]
S,D,W,sigma=pdf_weights(F)
A=add_temporal_edges(cdf_sparsify(W,GAMMA))      # sparsified + temporal (what the pipeline uses)
P=transition_matrix(A)
scores=propagate(P,Yw,seedw,RW_STEPS)            # per-class RW scores
n=len(idx); print(f'window: {n} frames | seeds={int(seedw.sum())} | classes present={int((Yw.sum(0)>0).sum())} | sigma={sigma:.3f}')

# single-action labels for coloring
def primary_gt(row):
    nz=np.where(row>0)[0]; return int(nz[0]) if len(nz) else -1   # -1 = background
gt_seed = {i: primary_gt(Yw[i]) for i in range(n) if seedw[i]}
pred_cls = scores.argmax(1)                       # single predicted class per frame (argmax)
active = scores.max(1) > 0                         # frames that received any mass


window: 100 frames | seeds=9 | classes present=4 | sigma=1.089


In [4]:
# ----- networkx graph + ONE fixed layout reused by all panels -----
G=nx.Graph()
G.add_nodes_from(range(n))
iu=np.triu_indices(n,k=1)
for a,b in zip(*iu):
    w=A[a,b]
    if w>0: G.add_edge(int(a),int(b),weight=float(w))
pos=nx.spring_layout(G,weight='weight',seed=LAYOUT_SEED,k=1.5/np.sqrt(n))
print(f'graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (after CDF sparsification)')


graph: 100 nodes, 3886 edges (after CDF sparsification)


In [5]:
# ----- Plotly drawing helpers: distinct colors + numeric label IDs -----
import colorsys

# Build colors only for class IDs that actually appear in this video.
# Evenly spaced hues avoid the old PALETTE modulo collisions.
_present_ids = sorted(
    {int(c) for c in gt_seed.values() if int(c) >= 0}
    | {int(pred_cls[i]) for i in range(n) if active[i] and int(pred_cls[i]) >= 0}
)

def _make_distinct_color_map(class_ids):
    m = max(1, len(class_ids))
    cmap = {}
    for j, cls in enumerate(class_ids):
        # Golden-ratio hue ordering improves separation between consecutive IDs.
        h = (j * 0.618033988749895) % 1.0
        s = 0.78
        v = 0.88
        r, g, b = colorsys.hsv_to_rgb(h, s, v)
        cmap[int(cls)] = f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'
    return cmap

CLASS_COLOR = _make_distinct_color_map(_present_ids)

def color_of(cls):
    cls = int(cls)
    return '#c9d6df' if cls < 0 else CLASS_COLOR.get(cls, '#808080')

def edge_trace():
    xs=[]; ys=[]
    for a,b in G.edges():
        xs += [pos[a][0], pos[b][0], None]
        ys += [pos[a][1], pos[b][1], None]
    return go.Scatter(
        x=xs, y=ys, mode='lines',
        line=dict(width=0.3, color='rgba(110,160,190,0.25)'),
        hoverinfo='none', showlegend=False
    )

def node_trace(colors, sizes, text=None):
    xs=[pos[i][0] for i in range(n)]
    ys=[pos[i][1] for i in range(n)]
    return go.Scatter(
        x=xs, y=ys, mode='markers',
        marker=dict(color=colors, size=sizes, line=dict(width=0.5, color='white')),
        hoverinfo='text',
        hovertext=text if text is not None else None,
        showlegend=False
    )

def make_fig(title, colors, sizes, node_labels=None, hover_text=None):
    data=[edge_trace(), node_trace(colors, sizes, text=hover_text)]
    fig=go.Figure(data=data)

    if node_labels:
        for i, txt in node_labels.items():
            fig.add_annotation(
                x=pos[i][0], y=pos[i][1],
                text=str(txt),
                showarrow=False,
                font=dict(color='black', size=11),
                bgcolor='rgba(255,255,255,0.75)',
                borderpad=1,
                yshift=13
            )

    fig.update_layout(
        title=title,
        template='plotly_white',
        width=650, height=650,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        margin=dict(l=10, r=10, t=40, b=10)
    )
    return fig


In [6]:
# ----- Panel 1: Input graph (CDF-sparsified), nodes neutral -----
fig1=make_fig(f'Input graph — {VIDEO} (N={n}, CDF-sparsified)',
              colors=['#4c78a8']*n, sizes=[6]*n)
fig1.show()
fig1.write_html(str(OUT_DIR/f'rwviz_1_input_{VIDEO}.html'))


In [7]:
# ----- Panel 2: Seeds on graph (distinct colors + numeric label_id) -----
colors2=['#dfe6ec']*n
sizes2=[5]*n
seed_lab={}
hover2=[f'node_id={i}' for i in range(n)]

for i, cls in gt_seed.items():
    cls = int(cls)
    colors2[i] = color_of(cls)
    sizes2[i] = 16
    seed_lab[i] = str(cls) if cls >= 0 else '-1'
    hover2[i] = f'node_id={i}<br>seed label_id={cls}'

fig2 = make_fig(
    f'Seeds on graph — {VIDEO} (numbers = label_id)',
    colors2,
    sizes2,
    node_labels=seed_lab,
    hover_text=hover2
)
fig2.show()

# Kaleido is unavailable in this runtime, so save an interactive HTML copy.
fig2.write_html(str(OUT_DIR / f'rwviz_2_seeds_{VIDEO}.html'))


In [8]:
# ----- Panel 3: Output after RW (distinct colors + numeric predicted label_id) -----
colors3=[]
sizes3=[]
output_lab={}
hover3=[]

for i in range(n):
    if seedw[i]:
        cls = int(gt_seed.get(i, -1))
        colors3.append(color_of(cls))
        sizes3.append(15)
        output_lab[i] = str(cls)
        hover3.append(f'node_id={i}<br>seed label_id={cls}')
    elif active[i]:
        cls = int(pred_cls[i])
        colors3.append(color_of(cls))
        sizes3.append(9)
        output_lab[i] = str(cls)
        hover3.append(f'node_id={i}<br>predicted label_id={cls}')
    else:
        colors3.append('#c9d6df')
        sizes3.append(6)
        hover3.append(f'node_id={i}<br>unreached/background')

fig3 = make_fig(
    f'After random walk — {VIDEO} (numbers = predicted label_id)',
    colors3,
    sizes3,
    node_labels=output_lab,
    hover_text=hover3
)
fig3.show()

# Kaleido is unavailable in this runtime, so save an interactive HTML copy.
fig3.write_html(str(OUT_DIR / f'rwviz_3_output_{VIDEO}.html'))

# Numeric label-ID color key
present = sorted(
    set(int(c) for c in gt_seed.values() if int(c) >= 0)
    | set(int(pred_cls[i]) for i in range(n) if active[i] and int(pred_cls[i]) >= 0)
)

print('label_id colors:')
for cls in present:
    print(f'  label_id={cls:>3}  color={color_of(cls)}')


label_id colors:
  label_id=  0  color=rgb(224,49,49)
  label_id=  1  color=rgb(49,100,224)
  label_id=  3  color=rgb(151,224,49)


### Optional — dense $W$ vs CDF-sparsified $A$ (sells the CDF step)


In [9]:
# side-by-side edge counts + a quick heatmap-free comparison
dense_edges=int((W>0).sum()-n)//2
sparse_edges=G.number_of_edges()
print(f'dense W edges: {dense_edges} | CDF-sparsified edges: {sparse_edges} '
      f'({100*sparse_edges/max(dense_edges,1):.1f}% kept)')
# heatmaps
import plotly.express as px
px.imshow(W,title='Dense W (all similarities)',color_continuous_scale='Blues').show()
px.imshow(A,title='CDF-sparsified A (+temporal)',color_continuous_scale='Blues').show()


dense W edges: 4950 | CDF-sparsified edges: 3886 (78.5% kept)
